In [ ]:
if len(df_wide) > 0 and len(col_reviewers) > 1:
    # Create a heatmap showing agreement patterns
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Heatmap 1: Reviewer annotations
    df_heatmap = df_wide[col_reviewers].copy()
    df_heatmap = df_heatmap.replace({'True': 1, 'False': 0, '': np.nan})
    
    sns.heatmap(df_heatmap, annot=False, cmap='RdYlGn_r', cbar_kws={'label': 'BabyDog'}, 
                ax=ax1, yticklabels=False)
    ax1.set_title(f'Annotation Pattern ({len(df_heatmap)} videos)')
    ax1.set_ylabel('Video ID')
    ax1.set_xlabel('Reviewer')
    
    # Chart 2: Agreement histogram
    agreement_counts = df_wide[col_reviewers].nunique(axis=1).value_counts().sort_index()
    ax2.bar(agreement_counts.index, agreement_counts.values, color=['green', 'orange', 'red'][:len(agreement_counts)])
    ax2.set_xlabel('Number of Different Answers')
    ax2.set_ylabel('Number of Videos')
    ax2.set_title('Agreement Distribution')
    ax2.set_xticks(agreement_counts.index)
    
    plt.tight_layout()
    plt.savefig('../../data/export/intercoder_agreement.png', dpi=100, bbox_inches='tight')
    print("✅ Saved visualization to data/export/intercoder_agreement.png")
    plt.show()

## Visualize Agreement Patterns

Generate visualizations showing inter-annotator agreement and disagreement distributions.

In [ ]:
# Count BabyDog appearances by reviewer
print(f"\n--- BabyDog Appearance Counts ---")
babydog_counts = (df_wide[col_reviewers] == 'True').sum()
total_reviewed = (df_wide[col_reviewers].isin(['True', 'False'])).sum()

for reviewer in col_reviewers:
    count = babydog_counts[reviewer]
    total = total_reviewed[reviewer]
    pct = count / total if total > 0 else 0
    print(f"{reviewer}: {count}/{total} videos ({pct:.1%})")

# Show disagreements
print(f"\n--- Disagreement Analysis ---")
df_disagreement = df_wide[~df_wide['unanimous']]
if len(df_disagreement) > 0:
    print(f"Total videos with disagreement: {len(df_disagreement)}\n")
    print("Videos with disagreement:")
    for idx, row in df_disagreement.iterrows():
        annotations = {rev: row[rev] for rev in col_reviewers}
        print(f"  {idx}: {annotations}")
else:
    print("Perfect agreement! All reviewers agree on all videos.")

## Generate Reliability Report

Create summary table showing BabyDog appearance counts by reviewer and identify disagreements.

In [ ]:
# Pivot to compare reviewers
df_wide = df_all.pivot_table(
    index='video_id',
    columns='reviewer',
    values='contains_babydog',
    aggfunc='first'
)

col_reviewers = df_wide.columns.tolist()
print(f"Videos in comparison: {len(df_wide)}\n")

# Calculate raw agreement
has_answer = df_wide[col_reviewers].isin(['True', 'False'])
has_same_answer = df_wide[col_reviewers].nunique(axis=1) == 1
df_wide['unanimous'] = has_same_answer & has_answer.all(axis=1)

print(f"--- Raw Agreement ---")
print(f"Unanimous agreement: {df_wide['unanimous'].mean():.1%}")
print(f"Videos with disagreement: {(~df_wide['unanimous']).sum()}\n")

# Calculate pairwise Cohen's Kappa
print(f"--- Pairwise Cohen's Kappa ---")
kappa_scores = {}
for i, rev1 in enumerate(col_reviewers):
    for rev2 in col_reviewers[i+1:]:
        # Only compare videos where both made a True/False annotation
        mask = (df_wide[rev1].isin(['True', 'False'])) & (df_wide[rev2].isin(['True', 'False']))
        if mask.sum() > 0:
            # Convert to binary (True=1, False=0)
            y1 = (df_wide.loc[mask, rev1] == 'True').astype(int)
            y2 = (df_wide.loc[mask, rev2] == 'True').astype(int)
            kappa = cohen_kappa_score(y1, y2)
            kappa_scores[f"{rev1} vs {rev2}"] = kappa
            print(f"{rev1} vs {rev2}: κ = {kappa:.3f} (n={mask.sum()})")

if kappa_scores:
    avg_kappa = np.mean(list(kappa_scores.values()))
    print(f"\nMean Cohen's Kappa: {avg_kappa:.3f}")
else:
    print("Not enough reviewers or complete annotations for Cohen's Kappa")

## Calculate Agreement Metrics

Compare annotators and compute agreement statistics, including raw agreement percentage and Cohen's Kappa for pairwise comparisons.

In [ ]:
# Load human annotations from data/export/human directory
human_labels_dir = Path('../../data/export/human/')

annotation_files = list(human_labels_dir.glob("*.csv"))

if annotation_files:
    print(f"Found {len(annotation_files)} annotation file(s)\n")
    
    # Combine all human annotations
    all_annotations = []
    for fn in annotation_files:
        df_human = pd.read_csv(fn, dtype=str, keep_default_na=False)
        all_annotations.extend(df_human.to_dict(orient="records"))
    
    df_all = pd.DataFrame(all_annotations)
    
    print(f"Total annotations: {len(df_all)}")
    print(f"Unique reviewers: {df_all['reviewer'].nunique()}")
    print(f"Reviewers: {', '.join(df_all['reviewer'].unique())}\n")
    print(df_all.head())
else:
    print("No annotation files found in:", human_labels_dir)
    print("\nTo run this analysis:")
    print("1. Launch the annotation tool: npm run dev")
    print("2. Enter your name and annotate all 50 videos")
    print("3. Download and save your CSV to:", human_labels_dir)

## Load Classification Data from Multiple Annotators

Load all CSV files from `data/export/human/` directory. Each file should contain annotations from one reviewer with columns: video_id, title, date, youtube_link, contains_babydog, interesting, reviewer, note

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from sklearn.metrics import cohen_kappa_score, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded successfully")

# Intercoder Reliability Analysis

Analyze agreement between multiple human annotators classifying BabyDog appearances in 50 press briefings. This notebook calculates Cohen's Kappa, Fleiss' Kappa, and other agreement metrics to assess annotation quality and consistency.